# DOE that shows real metric improvement: banking77 SFT

This notebook is a **thin demo**: every step calls into the tested
`geap_tuning` package. It is the discriminating counterpart of
[`10_doe.ipynb`](10_doe.ipynb). See
[`docs/notes/doe-and-visualization.md`](../docs/notes/doe-and-visualization.md)
and [`docs/notes/banking77-dataset.md`](../docs/notes/banking77-dataset.md).

**Why a second SFT DOE?** The demo sweep in `10_doe.ipynb` (Experiment
`geap-doe-sft`) **saturates at accuracy = 1.0 in every grid cell** — its
5-intent support-ticket set is too easy, so no hyperparameter can be told
apart. Here we cross the **same** small SFT grid (`epochs` × `adapter_size`,
4 runs) on **banking77** (77 fine-grained banking intents; PolyAI/banking77,
CC-BY-4.0), which has real headroom. We also score the **untuned base model**
as a baseline, so the result is a genuine **before → after** story.

Metric values are our own offline `run_eval` on the held-out test split (the
Layer-1 Monitor loss curves are console-only, not SDK-fetchable). Charts need
the optional viz group (`uv sync --group viz`).

> **Requires live GCP and incurs tuning cost:** 4 cheap `gemini-2.5-flash-lite`
> SFT jobs + baseline/eval inference. Have a real `.env` and `gcloud auth` in
> place; tuning is **regional-only**, so keep the tuning/Experiments region
> aligned.

In [ ]:
from geap_tuning.config import genai_client, load_config
from geap_tuning.doe import SweepConfig

EXPERIMENT_NAME = "geap-doe-banking77"
BASE_MODEL = "gemini-2.5-flash-lite"
PER_CLASS = {"train": 10, "val": 2, "test": 5}  # 770 / 154 / 385 examples
SWEEP = SweepConfig(
    name="banking77",
    base_model=BASE_MODEL,
    grid={"epochs": [2, 8], "adapter_size": [4, 16]},  # 4 runs
)

cfg = load_config()
client = genai_client(cfg)  # tuning is regional-only
cfg

## 1. Build and stage the banking77 SFT dataset

`build_banking_dataset` samples a balanced subset (train/val carved disjoint
per label from the train split; test from the held-out test split) and writes
three JSONL files. The two CSVs are fetched via stdlib and cached under
`datasets/banking77_sft/raw` (no extra dependency); pass `csv_dir=` for a
pre-downloaded copy offline.

In [ ]:
from geap_tuning.gcs import upload_file
from geap_tuning.sft.banking import build_banking_dataset

paths = build_banking_dataset("../datasets/banking77_sft", per_class=PER_CLASS)
train_uri = upload_file(paths["train"], f"{cfg.bucket}/doe_banking77/train.jsonl")
val_uri = upload_file(paths["val"], f"{cfg.bucket}/doe_banking77/val.jsonl")
train_uri, val_uri

## 2. Held-out test records + the shared label space

Read the sampled test JSONL back for offline scoring, and derive the 77 labels
+ the system instruction from the train CSV so the scorer constrains every
reply to a valid intent (fair for the untuned baseline too).

In [ ]:
import json
from pathlib import Path

from geap_tuning.sft.banking import banking_labels, build_system_instruction, load_pairs_from_csv

test_records = [json.loads(line) for line in Path(paths["test"]).read_text().splitlines()]
labels = banking_labels(load_pairs_from_csv("../datasets/banking77_sft/raw/train.csv"))
system_instruction = build_system_instruction(labels)
len(labels), len(test_records)

## 3. Point Vertex AI Experiments at the tuning region

Call `init_experiment` **before** logging any run so the baseline and every
sweep run attach to the same experiment (no TensorBoard needed for summary
metrics).

In [ ]:
from geap_tuning.experiments import init_experiment

init_experiment(EXPERIMENT_NAME, project=cfg.project, location=cfg.location)

## 4. Define the offline scorer

`run_sweep` calls `evaluate_fn(endpoint)` per run. We supply the label list via
the system instruction and canonicalize each reply with
`parse_banking_prediction` before scoring on the held-out test split.

In [ ]:
from geap_tuning.inference import generate
from geap_tuning.sft.banking import parse_banking_prediction
from geap_tuning.sft.evaluate import run_eval


def evaluate_fn(endpoint: str) -> dict:
    def predict(user_text: str) -> str:
        reply = generate(client, endpoint, user_text, system_instruction=system_instruction)
        return parse_banking_prediction(reply, labels)

    return run_eval(test_records, predict_fn=predict)

## 5. Baseline (before): score the untuned base model

The untuned endpoint is just the base-model name. We log it as its own
Experiments run (`epochs`/`adapter_size` = 0) so it lines up next to the tuned
runs in the comparison.

In [ ]:
from geap_tuning.experiments import log_summary_metrics, track_run

base_metrics = evaluate_fn(BASE_MODEL)
with track_run(
    f"geap-doe-{SWEEP.name}-baseline",
    params={"base_model": BASE_MODEL, "epochs": 0, "adapter_size": 0},
):
    log_summary_metrics(
        {"accuracy": base_metrics["accuracy"], "macro_f1": base_metrics["macro_f1"]}
    )
round(base_metrics["accuracy"], 3), round(base_metrics["macro_f1"], 3)

## 6. Sweep (after): run the grid

Each grid point reuses a matching job if one exists (cost control), else
launches a `gemini-2.5-flash-lite` SFT job, waits, scores, and logs to
Experiments.

In [ ]:
from geap_tuning.doe import run_sweep

results = run_sweep(
    client,
    SWEEP,
    train_uri=train_uri,
    val_uri=val_uri,
    evaluate_fn=evaluate_fn,
    experiment=EXPERIMENT_NAME,
)
[
    (r.spec.name, round(r.metrics["accuracy"], 3), "reused" if r.reused else "launched")
    for r in results
]

## 7. Aggregate, pick the winner, report the improvement

Prepend the baseline as its own row, then flatten the tuned runs with
`aggregate_results`. `select_best_run` picks the headline metric (accuracy);
the improvement is `best − baseline`.

In [ ]:
from geap_tuning.doe import HEADLINE_METRIC, METRICS_BY_METHOD, aggregate_results, select_best_run

metrics = METRICS_BY_METHOD["SFT"]
baseline_row = {
    "run": "baseline",
    "base_model": BASE_MODEL,
    "epochs": 0,
    "adapter_size": 0,
    "accuracy": base_metrics["accuracy"],
    "macro_f1": base_metrics["macro_f1"],
}
rows = [baseline_row, *aggregate_results(results, metrics=metrics)]
by_name = {r.spec.name: r.metrics for r in results}
best = select_best_run(by_name, metric=HEADLINE_METRIC["SFT"])
print("best run:", best)
print(f"improvement over baseline: {by_name[best]['accuracy'] - base_metrics['accuracy']:+.3f}")
rows

## 8. Compare baseline + runs visually

Requires the viz group (`uv sync --group viz`). One cluster per run (baseline
first) with a bar per metric.

In [ ]:
from geap_tuning.viz import plot_grouped_metric_bars

plot_grouped_metric_bars(rows, metrics=metrics)

## 9. The same runs, read back from Experiments

`experiment_dataframe` returns a pandas table matching **Agent Platform Studio
→ Experiments** — the baseline plus the 4 tuned runs, now with **varying**
metrics (the discrimination the demo sweep lacked).

In [ ]:
from geap_tuning.experiments import experiment_dataframe

experiment_dataframe(EXPERIMENT_NAME)

## Next steps

Read the tracked runs back with zero tuning cost in
[`11_multi_run_viz.ipynb`](11_multi_run_viz.ipynb)
(`--experiment geap-doe-banking77`), or contrast with the saturated demo sweep
in [`10_doe.ipynb`](10_doe.ipynb). See
[`docs/notes/banking77-dataset.md`](../docs/notes/banking77-dataset.md) for the
dataset details.